> ### **Building with Sarvam**
>
> **An open teaching kit for the Sarvam AI stack.**
>
> Notebook maintained by **Dr. Bhaveshkumar C. Dharmani** — Founder & AI Mentor, AIVidhya4Sarvam.
PhD (ICT), DA-IICT Gandhinagar · https://www.aividhya.in/ · https://www.aividhya4sarvam.in/ · bhavesh@aividhya.in  · https://www.linkedin.com/in/bhaveshdharmani/
>
> Drafted with AI assistance and stress-tested cell by cell in live workshop sessions. Runs against your own Sarvam API key from dashboard.sarvam.ai, with a live ₹ cost meter after every call.
>
> Apache 2.0 · Issues and PRs welcome at github.com/dharmanibc/building-with-sarvam.S

<div style="background:#12172E;color:#fff;padding:20px 24px;border-radius:8px">
<div style="color:#FF8A3D;font-size:12px;letter-spacing:2px;font-weight:700">LAB 01 · THE LIVE SESSION LAB</div>
<div style="font-size:26px;font-weight:700;margin-top:6px">Every API in one script</div>
<div style="color:#FFB37A;font-size:14px;margin-top:8px">Language ID → Translate → Transliterate → TTS → STT → cost report</div>
</div>

**Time:** 30 min &nbsp;·&nbsp; **Est. cost:** ≈ ₹4.20 &nbsp;·&nbsp; **Prereq:** Lab 00

## The point of this lab

One file that touches the **entire live surface** of the platform and prints what
each call cost. Step 4 produces an audio file you can play — hearing your own
language spoken by code you just ran is the moment this stops being abstract.

> **Teaching note.** Run this cell by cell in front of the room. The chat filling
> with people's generated audio files is your best social proof of the session.

In [1]:
# ── Standard lab header. Run this first in every notebook. ─────────────────
import os, sys, json, time, math, wave, io
from pathlib import Path

# pip install sarvamai python-dotenv

from dotenv import load_dotenv, find_dotenv
# Finds your key without hardcoding anyone's filesystem. Tried in order:
#   1. SARVAM_API_KEY already set in the environment
#   2. the file named by SARVAM_ENV_FILE, if you set that variable
#   3. a .env beside this notebook, or in any parent folder
load_dotenv(os.environ.get("SARVAM_ENV_FILE") or find_dotenv(usecwd=True))

API_KEY = os.environ.get("SARVAM_API_KEY")
assert API_KEY, (
    "SARVAM_API_KEY not found.\\n"
    "Create a .env next to this notebook containing:  SARVAM_API_KEY=sk_...\\n"
    "or point SARVAM_ENV_FILE at an existing env file.\\n"
    "Free key + Rs 1000 credit: https://indus.sarvam.ai/"
)

from sarvamai import SarvamAI

client = SarvamAI(api_subscription_key=API_KEY)
DATA = Path("./data"); DATA.mkdir(exist_ok=True)
OUT  = Path("./out");  OUT.mkdir(exist_ok=True)
print("SDK ready ·", sys.version.split()[0])

SDK ready · 3.13.9


In [2]:
# ── The ₹ meter, imported ─────────────────────────────────────────────────
# Lab 00 writes cost_meter.py next to these notebooks. If this import fails,
# run Lab 00 once — it is the only lab that defines the meter.
try:
    from cost_meter import CostMeter, RATES, FREE_CREDIT
except ImportError:
    raise ImportError(
        "cost_meter.py not found.\n"
        "Run 00_Setup_and_the_Cost_Meter.ipynb once — its last section writes "
        "cost_meter.py into this folder, and every other lab imports it from there."
    )

cost = CostMeter()
print(f"cost meter armed · rates dated Aug 2026 · ₹{FREE_CREDIT:.0f} free credit")


cost meter armed · rates dated Aug 2026 · ₹1000 free credit


### 1 · Language ID — what language is this, and in what script?

In [3]:
SAMPLES = [
    "मेरा EMI due date क्या है",        # Hindi, Devanagari, code-mixed
    "mera EMI due date kya hai",       # Hindi, romanised
    "What is my EMI due date?",        # English
    "எனது கடன் தவணை எப்போது?",          # Tamil
    "Maara EMI ni due date kai chhe?"
]

for s in SAMPLES:
    r = client.text.identify_language(input=s)
    cost.text(len(s), "lid")
    print(f"{r.language_code:>8}  {getattr(r, 'script_code', '—'):>8}   {s}")

   hi-IN      Deva   मेरा EMI due date क्या है
   hi-IN      Latn   mera EMI due date kya hai
   en-IN      Latn   What is my EMI due date?
   ta-IN      Taml   எனது கடன் தவணை எப்போது?
   gu-IN      Latn   Maara EMI ni due date kai chhe?


**Why this matters:** at ₹3.50 per 10,000 characters, LID is cheap enough to run on
*every* inbound message as a router. Most people never think to use it.

### 2 · Translate — English ⇄ Hindi

In [4]:
EN = "Your loan instalment of Rs 12,500 is due on the 15th of this month."
MY_LANGUAGE = "hi-IN"  

my_lan = client.text.translate(
    input=EN,
    source_language_code="en-IN",
    target_language_code=MY_LANGUAGE, #"gu-IN",
    model="mayura:v1",
    mode="formal",            # classic-colloquial (Heritage) | modern-colloquial (Contemporary) | code-mixed (Bilingual) | formal (Official)
)
cost.text(len(EN), "translate")
print("HI :", my_lan.translated_text)


HI : आपका 12,500 रुपये का ऋण किश्त इस महीने की 15 तारीख को देय है।


In [5]:
my_lan

TranslationResponse(request_id='20260828_0eb9f443-a7a5-4fc9-8e9b-0a3eec902939', translated_text='आपका 12,500 रुपये का ऋण किश्त इस महीने की 15 तारीख को देय है।', source_language_code='en-IN')

In [6]:
back = client.text.translate(
    input=my_lan.translated_text,
    source_language_code=MY_LANGUAGE, #"hi-IN",
    target_language_code="en-IN",
    model="mayura:v1",
)
cost.text(len(my_lan.translated_text), "translate")
print("EN :", back.translated_text)

EN : Your loan payment of 12,500 rupees is due on the 15th of this month.


In [7]:
EN = "Your loan instalment of Rs 12,500 is due on the 15th of this month."
# MY_LANGUAGE = "hi-IN"  

my_lan = client.text.translate(
    input=EN,
    source_language_code="en-IN",
    target_language_code=MY_LANGUAGE, #"gu-IN",
    model="mayura:v1",
    mode="modern-colloquial",  
    # classic-colloquial (Heritage) | modern-colloquial (Contemporary) | code-mixed (Bilingual) | formal (Official))
)
cost.text(len(EN), "translate")
print("HI :", my_lan.translated_text)

HI : आपका Rs. 12,500 का loan instalment इस month की 15th को due है।


In [8]:
EN = "Your loan instalment of Rs 12,500 is due on the 15th of this month."
# MY_LANGUAGE = "hi-IN"  

my_lan = client.text.translate(
    input=EN,
    source_language_code="en-IN",
    target_language_code=MY_LANGUAGE, #"gu-IN",
    model="mayura:v1",
    mode="code-mixed",         # classic-colloquial (Heritage) | modern-colloquial (Contemporary) | code-mixed (Bilingual) | formal (Official)       
)
cost.text(len(EN), "translate")
print("HI :", my_lan.translated_text)

HI : आपका Rs. 12,500 का loan instalment इस month की 15th को due है।


### 3 · Transliterate — same sound, different script

In [9]:
tr = client.text.transliterate(
    input=my_lan.translated_text,
    source_language_code=MY_LANGUAGE, #"gu-IN",
    target_language_code=MY_LANGUAGE, #"gu-IN",
    spoken_form=True,
)
cost.text(len(my_lan.translated_text), "transliterate")
print("Devanagari :", my_lan.translated_text)
print("Roman      :", tr.transliterated_text)

Devanagari : आपका Rs. 12,500 का loan instalment इस month की 15th को due है।
Roman      : आपका बारह हज़ार पाँच सौ रुपीज़ का लोन इंस्टॉलमेंट इस मंथ की वन फाइव टी एच को ड्यू है।


### 4 · Text to speech — **the moment**

Change `MY_LANGUAGE` to your own. Run it. Play the file.

In [10]:
from sarvamai.play import save

# MY_LANGUAGE = "hi-IN"      # ← change me: ta-IN bn-IN te-IN mr-IN gu-IN kn-IN ml-IN pa-IN od-IN
MY_TEXT     = tr.transliterated_text #my_lan.translated_text
SPEAKER     = "pooja"     # "anushka"  --> not available for bulbul:v3 and v2 is depricated

audio = client.text_to_speech.convert(
    text=MY_TEXT,
    language_code=MY_LANGUAGE,
    model="bulbul:v3",         # v2 = ₹15/10k. v3 = ₹30/10k. Both works. See Lab 03.
    speaker=SPEAKER,
)
save(audio, str(OUT / "my_language.wav"))
cost.tts(len(MY_TEXT), v3=False)
print("saved →", OUT / "my_language.wav")

saved → out/my_language.wav


In [11]:
# Play it right here in the notebook
from IPython.display import Audio, display

file_path = str(OUT / "my_language.wav")
display(Audio(filename=file_path))

# p = OUT / f"voice_{spk}.wav"; save(a, str(p));

### 5 · Speech to text — transcribe it back, all five modes

In [12]:
MODES = ["transcribe", "translate", "verbatim", "translit", "codemix"]
results = {}

for m in MODES:
    with open(OUT / "my_language.wav", "rb") as f:
        r = client.speech_to_text.transcribe(
            file=f, 
            model="saaras:v3", 
            language_code=MY_LANGUAGE, 
            mode=m,
        )
    results[m] = r.transcript
    cost.stt(4.0)          # ~4s clip; replace with real duration in Lab 02
    print(f"{m:>11} │ {r.transcript}")

 transcribe │ आपका बारह हज़ार पाँच सौ रुपीज़ का लोन इंस्टॉलमेंट इस मंथ की वन फाइव टी एच को ड्यू है।
  translate │ Your loan installment of twelve thousand five hundred rupees is due on the fifteenth of this month.
   verbatim │ आपका बारह हज़ार पाँच सौ रुपीज़ का लोन इंस्टॉलमेंट इस मंथ की वन फाइव टी एच को ड्यू है।
   translit │ Aapka baarah hazaar paanch sau rupees ka loan installment is month ki ek sau TH ko due hai
    codemix │ आपका बारह हज़ार पाँच सौ रुपीज़ का लोन इंस्टॉलमेंट इस मंथ की वन फाइव टी एच को ड्यू है।


In [13]:
r = client.speech_to_text.transcribe(
            file= open(OUT / "my_language.wav", "rb"),
            model="saaras:v3", 
            mode="transcribe",
        )
print("WITHOUT language_code :", r.transcript)

WITHOUT language_code : आपका बारह हज़ार पाँच सौ रुपीज़ का लोन इंस्टॉलमेंट इस मंथ की वन फाइव टी एच को ड्यू है।


### 6 · The cost report

In [14]:
cost.report()

lid          ₹   0.0087  25 chars
lid          ₹   0.0087  25 chars
lid          ₹   0.0084  24 chars
lid          ₹   0.0080  23 chars
lid          ₹   0.0109  31 chars
translate    ₹   0.1340  67 chars
translate    ₹   0.1220  61 chars
translate    ₹   0.1340  67 chars
translate    ₹   0.1340  67 chars
transliterate ₹   0.1240  62 chars
TTS          ₹   0.1245  83 chars
STT          ₹   0.0333  4.0s
STT          ₹   0.0333  4.0s
STT          ₹   0.0333  4.0s
STT          ₹   0.0333  4.0s
STT          ₹   0.0333  4.0s
TOTAL        ₹   0.9840
              (₹1000 free credit → ₹999.02 left)
              Estimated from published rates; actual billing usually lower


0.9839666666666667

---
## ✅ Checkpoint

- [ ] Five modes produced five genuinely different transcripts
- [ ] You measured the WER damage from a missing `sample_rate`
- [ ] A batch job with diarization completed and you read the speaker turns
- [ ] Streaming produced partial results before the audio finished

## 🧪 Try this

1. Record 30 seconds of yourself, in your own dialect. Run all five modes. Where does it fail?
2. Downsample to 8 kHz and re-measure WER. How much accuracy does telephony cost you?
3. Time the batch job end to end. How would that latency change your product design?
4. Build a 10-file eval set from real audio and compute mean WER. **This is the artefact
   that wins enterprise pilots** — a documented accuracy number on *their* data.

### **23-language set (Saaras v3, Sarvam Translate, Sarvam Vision)**
**Language	Code		              Language	Code** 

Hindi	hi-IN		                  Assamese	as-IN

Bengali	bn-IN		                  Urdu	ur-IN

Kannada	kn-IN		                  Nepali	ne-IN

Malayalam	ml-IN		              Konkani	kok-IN

Marathi	mr-IN		                  Kashmiri	ks-IN

Odia	od-IN		                  Sindhi	sd-IN

Punjabi	pa-IN		                  Sanskrit	sa-IN

Tamil	ta-IN		                  Santali	sat-IN

Telugu	te-IN		                  Manipuri	mni-IN

English	en-IN		                  Bodo	brx-IN

Gujarati	gu-IN		                  Maithili	mai-IN

Dogri	doi-IN

### **11-language set (Bulbul v3, Mayura, Sarvam-105B, Saarika v2.5)**
**Language	Code		Language	Code**

Hindi	hi-IN		Kannada	kn-IN

Bengali	bn-IN		Malayalam	ml-IN

Tamil	ta-IN		Marathi	mr-IN

Telugu	te-IN		Punjabi	pa-IN

Gujarati	gu-IN		Odia	od-IN

English	en-IN			